In [8]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from pathlib import Path

In [59]:
data_path = Path.cwd() / "../data"
os.chdir(data_path)
print(os.getcwd())

file_names = [p.name[:-5] for p in data_path.iterdir() if p.is_file()]
print(file_names)

data = {}
for name in file_names:
    with open(data_path / f'{name}.json', encoding='utf-8') as f:
        data[name] = json.load(f)
bus = data['bus_params']

for idx in range(len(bus[0]["energy_price_DKK_per_kWh"])):
    print(f"Hour {idx} price: ", bus[0]["energy_price_DKK_per_kWh"][idx])

#Import/Export prices
print(f'Import price: ', bus[0]['import_tariff_DKK_per_kWh'], '\nExport price: ',
 bus[0]['export_tariff_DKK_per_kWh'])

/home/asgoodasbread/Desktop/Optimization for modern power systems/Assignment_1/assigment1_group1/data
['appliance_params', 'bus_params', 'params_Q1_caseA', 'params_Q1_caseB', 'params_Q2_linear', 'params_Q2_quadratic', 'params_Q3', 'params_Q3_battery']
Hour 0 price:  1.1
Hour 1 price:  1.05
Hour 2 price:  1.0
Hour 3 price:  0.9
Hour 4 price:  0.85
Hour 5 price:  1.01
Hour 6 price:  1.05
Hour 7 price:  1.2
Hour 8 price:  1.4
Hour 9 price:  1.6
Hour 10 price:  1.5
Hour 11 price:  1.1
Hour 12 price:  1.05
Hour 13 price:  1.0
Hour 14 price:  0.95
Hour 15 price:  1.0
Hour 16 price:  1.2
Hour 17 price:  1.5
Hour 18 price:  2.1
Hour 19 price:  2.5
Hour 20 price:  2.2
Hour 21 price:  1.8
Hour 22 price:  1.4
Hour 23 price:  1.2
Import price:  0.5 
Export price:  0.4


In [67]:
P_price_per_kwh = bus[0]['energy_price_DKK_per_kWh']
export_price = bus[0]['export_tariff_DKK_per_kWh']
import_price = bus[0]['import_tariff_DKK_per_kWh']

hours_u_greater_p_imp = []
hours_u_lesser_p_exp = []

for h in range(24):
    pt_imp = P_price_per_kwh[h] + export_price
    pt_exp = P_price_per_kwh[h] - import_price

    #Consumption utility for problem 1
    u = data['appliance_params']['loads'][0]['consumption_utility_DKK_per_kWh']

    if u > pt_imp:
        hours_u_greater_p_imp.append(h)
    elif u < pt_exp:
        hours_u_lesser_p_exp.append(h)

print('Import: ', hours_u_greater_p_imp, '\n', 'Export: ', hours_u_lesser_p_exp, sep ='')

Import: [2, 3, 4, 5, 13, 14, 15]
Export: [18, 19, 20]


In [78]:
c_pv_a = data['appliance_params']['DERs'][0]['marginal_cost_DKK_per_kWh']
c_pv_b = data['appliance_params']['DERs'][1]['marginal_cost_DKK_per_kWh']
u = data['appliance_params']['loads'][0]['consumption_utility_DKK_per_kWh']

print('Case A: ', round(u - c_pv_a, 2))
print('Case B: ', round(u - c_pv_b, 2))

Case A:  0.02
Case B:  -0.04


In [88]:
min_energy = data['appliance_params']['loads'][3]['min_total_energy_per_day_kWh']

ref_lod_cap_factor = data['appliance_params']['loads'][
    3]['reference_load_capacity_factor']
max_load_kWh_per_hour = data['appliance_params']['loads'][3]['max_load_KWh_per_hour']

lt_ref = round(sum([max_load_kWh_per_hour * ref_lod_cap_factor[x]
               for x in range(len(ref_lod_cap_factor))]), 2)
print('lt_ref: ', lt_ref)

lt_ref:  32.34


In [106]:
# PV_01_A 0 -> max_power_kW
# ref_load_cap_factor_fl2l = data['appliance_params']['loads'][1]['reference_load_capacity_factor']
# ref_load_cap_factor_fl2q = data['appliance_params']['loads'][2]['reference_load_capacity_factor']
# ref_load_cap_factor_fl3 = data['appliance_params']['loads'][3]['reference_load_capacity_factor']

# max_load_kWh_per_hour = data['appliance_params']['loads'][0]['max_load_KWh_per_hour']

max_power_kW = data['appliance_params']['DERs'][0]['max_power_kW']
hourly_profile_ratio = data['appliance_params']['DERs'][0]['hourly_profile_ratio']

pv_available = [max_power_kW * r for r in hourly_profile_ratio]
print('PV hours:', [h for h in range(24) if pv_available[h] > 0])
print('Total PV:', round(sum(pv_available), 2))


for h in [18, 19]:
    p_imp = P_price_per_kwh[h] + import_price
    p_exp = P_price_per_kwh[h] - export_price
    print(f'Hour {h}: PV = {pv_available[h]:.2f} kWh, p = {P_price_per_kwh[h]:.2f}, ' 
          f'p_imp = {p_imp:.2f}, p_exp = {p_exp:.2f}')


PV hours: [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Total PV: 26.94
Hour 18: PV = 1.50 kWh, p = 2.10, p_imp = 2.60, p_exp = 1.70
Hour 19: PV = 1.50 kWh, p = 2.50, p_imp = 3.00, p_exp = 2.10
